# LoRA fine-tune — Qwen3-VL-2B Persian line OCR (Colab T4)

Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom. Everything persistent goes to Google Drive under `MyDrive/exam-ocr/`, so a disconnect resumes from the last checkpoint when you re-run the training cell.

Step-by-step: `training/lora/README.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/exam-ocr'
import os; os.makedirs(DRIVE, exist_ok=True)

## Code
If the repository is private, replace the URL with `https://<TOKEN>@github.com/alireza-taheriF/exam-ocr.git` (a fine-grained read-only token).

In [ ]:
BRANCH = 'claude/plan-90-feedback-loop'
!rm -rf /content/exam-ocr && git clone -q -b $BRANCH https://github.com/alireza-taheriF/exam-ocr.git /content/exam-ocr
%cd /content/exam-ocr
!git log --oneline -1

In [ ]:
!pip -q install 'transformers>=5.0' peft accelerate datasets python-dotenv

## Local data (optional)
Upload `exam_ocr_lora_data.zip` (from `training/lora/make_bundle.py`) to `MyDrive/exam-ocr/`. Without it, training uses only the public datasets.

In [ ]:
import glob, os
BUNDLE = f'{DRIVE}/exam_ocr_lora_data.zip'
LOCAL = []
if os.path.exists(BUNDLE):
    !rm -rf /content/data && mkdir -p /content/data && unzip -q -o $BUNDLE -d /content/data
    !cat /content/data/bundle_summary.json
    LOCAL = sorted(glob.glob('/content/data/manifests/*.jsonl'))
print('local manifests:', LOCAL)

## Train
`NAME` is the adapter version. Defaults fit a free T4 (~15 GB): batch 2 × grad-accum 8, fp16 AMP with fp32 LoRA weights, gradient checkpointing, image long side ≤ 768 px. If you hit CUDA OOM, set `--batch-size 1 --grad-accum 16` or `--max-long-side 512`.

The first lines of the log print the HF dataset columns that were detected; if a column was guessed wrong, pass `--hw-image-col/--hw-text-col/--hw-bbox-col`.

In [ ]:
NAME = 'lora_v2'
OUT = f'{DRIVE}/{NAME}'
local_args = ' '.join(LOCAL)
!python training/lora/train_qwen_lora.py --out $OUT --device cuda \
    --hf-handwriting 6000 --replay-ratio 0.5 --data-dir /content/ocr_data \
    --epochs 1 --batch-size 2 --grad-accum 8 --lr 1e-4 --lora-r 16 \
    {'--local ' + local_args if LOCAL else ''}

In [ ]:
import json
meta = json.load(open(f'{OUT}/training_meta.json'))
print({k: meta[k] for k in ['n_train','train_counts','steps','loss_first','loss_last','val_cer_before','val_cer_after','minutes']})

## Package the adapter for download
Download `MyDrive/exam-ocr/<NAME>_adapter.zip` and unzip it into `models/<NAME>/` on the Mac.

In [ ]:
!cd $OUT/.. && rm -f {NAME}_adapter.zip && zip -q -r {NAME}_adapter.zip {NAME} -x '{NAME}/checkpoint-*/*' && ls -lh {NAME}_adapter.zip

## Then, on the Mac (not here)
```bash
unzip ~/Downloads/lora_v2_adapter.zip -d models/
HF_HUB_OFFLINE=1 .venv-qwen/bin/python scripts/eval_gate.py --label lora_v2 --adapter models/lora_v2 --register
```
Activate with `QWEN_ADAPTER_PATH=models/lora_v2` **only** if the registry says `champion`.